# Analyse Exploratoire v2 — Modele CLV Nouvelles Acquisitions

Version 2 structuree autour des **4 questions metier**, avec ventilation **exhaustive**
selon 6 caracteristiques client (cohorte / canal / tranche de revenu / residence /
age / CSP).

> Donnees propres (`clean_data/*.parquet`) produites par `data_preparation.ipynb`
> (inclut desormais **canal** sur STOCK et **clean_upstream** BDL/BPL).

## Plan
0. **Fondations** — chargement, panel famille, tenure, moteur de ventilation `break_by`
1. **Qui sont nos nouveaux clients ?** — acquisitions par caracteristique
2. **Activite des premiers mois** — churn, upstream, equipement & encours M+3/6/12, trajectoires
3. **Quel revenu genèrent-ils ?** — PnB M+3/6/12, PnB Y1→Y5, concentration, par produit
4. **Comment calibrer le modèle ?** — equipement/encours/PnB M+3 vs M+12 vs Y5, marges, deciles

> Code en anglais, narration en francais. Exercice refait **hors mineurs / hors etudiants** via `EXCLUDE_*`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

CLEAN_DIR = Path("../clean_data")
FIG_DIR = Path("../eda_figures_v2")
FIG_DIR.mkdir(exist_ok=True)

EPOCH_YEAR = 1994
def idx_to_year(idx): return EPOCH_YEAR + (int(idx) - 1) // 12
def idx_to_ym(idx):   return (EPOCH_YEAR + (int(idx) - 1) // 12, (int(idx) - 1) % 12 + 1)

NBI_COL = "revenus_eur_avec_liquidity"

FAM_ORDER  = ["epargne", "invest", "credit_immo", "autres_cr", "cartes"]
FAM_COLORS = {"epargne": "#06B6D4", "invest": "#7C3AED", "credit_immo": "#EC4899",
              "autres_cr": "#F59E0B", "cartes": "#84CC16"}
FAM_LABELS = {"epargne": "Epargne", "invest": "Investissements", "credit_immo": "Credit immo",
              "autres_cr": "Autres credits", "cartes": "Autre PNB"}
FAMILY_ENCOURS = {
    "epargne":      ("savings",   ["avoirs_eur"]),
    "invest":       ("invest",    ["cp_depots_a_terme", "cp_depots_en_cc_epar_liq",
                                   "cp_depots_en_cc_placement", "cp_depots_titres", "cp_epargne_a_vue"]),
    "invest_banca": ("banca",     ["capitaux_productifs_eur"]),
    "credit_immo":  ("pret_immo", ["credits_utilises_eur"]),
    "autres_cr":    ("credit",    ["credits_utilises_eur"]),
}

plt.rcParams.update({"figure.figsize": (13, 6), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titlesize": 12, "axes.labelsize": 10, "figure.dpi": 100})
sns.set_palette("muted")
PURPLE, CYAN, PINK = "#7C3AED", "#06B6D4", "#EC4899"

def save_fig(name, fig=None):
    fig = fig or plt.gcf()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()

print("Setup OK — figures ->", FIG_DIR.resolve())

## 0. Fondations

Chargement, panel famille, axe d'anciennete (`tenure`), et surtout le **moteur de
ventilation** : `break_by(metric)` trace n'importe quelle metrique client selon les
6 caracteristiques d'un coup. Toutes les vues "selon les differentes caracteristiques"
en decoulent.

In [ ]:
def load(name):
    p = CLEAN_DIR / f"clean_{name}.parquet"
    if not p.exists():
        print(f"  MANQUANT: {p.name}"); return None
    df = pd.read_parquet(p)
    print(f"  {name:<12}: {len(df):>10,} lignes x {df.shape[1]} cols")
    return df

print("Chargement:")
stock     = load("stock")
flux      = load("flux")
savings   = load("savings")
invest    = load("invest")
banca     = load("banca")
cartes    = load("cartes")
credit    = load("credit")
pret_immo = load("pret_immo")
upstream  = load("upstream")     # scope BDL/BPL (id, periode, scope) — peut etre None

PRODUCTS = {"savings": savings, "invest": invest, "banca": banca,
            "cartes": cartes, "credit": credit, "pret_immo": pret_immo}
HAS_CANAL    = stock is not None and "canal" in stock.columns
HAS_UPSTREAM = upstream is not None and len(upstream) > 0
print(f"\ncanal dispo: {HAS_CANAL} | upstream dispo: {HAS_UPSTREAM}")

In [ ]:
# Periode d'acquisition, tenure, alias NBI
acq = stock.groupby("identifiant_tiers")["periode"].min().rename("acq_period")

def add_tenure(df):
    if df is None: return None
    df = df.merge(acq, on="identifiant_tiers", how="left")
    df["tenure"] = (df["periode"] - df["acq_period"]).astype("Int64")
    return df

stock = add_tenure(stock); flux = add_tenure(flux)
for k in list(PRODUCTS): PRODUCTS[k] = add_tenure(PRODUCTS[k])
savings, invest, banca, cartes, credit, pret_immo = (PRODUCTS[k] for k in
    ["savings", "invest", "banca", "cartes", "credit", "pret_immo"])
if NBI_COL in flux.columns: flux["nbi"] = flux[NBI_COL]

max_tenure = flux.groupby("identifiant_tiers")["tenure"].max()
cohort_of  = acq.map(idx_to_year)
print(f"Clients: {stock['identifiant_tiers'].nunique():,} | anciennete mediane {int(max_tenure.median())} mois")

In [ ]:
# ── Panel famille : encours + NBI par (client, periode, famille) ─────────────
def agg_encours(df, cols):
    if df is None: return None
    cols = [c for c in cols if c in df.columns]
    if not cols: return None
    tmp = df[["identifiant_tiers", "periode"]].copy()
    tmp["encours"] = df[cols].sum(axis=1, min_count=1)
    return tmp.groupby(["identifiant_tiers", "periode"], as_index=False)["encours"].sum()

def agg_nbi(df):
    if df is None or NBI_COL not in df.columns: return None
    tmp = df[["identifiant_tiers", "periode", NBI_COL]].rename(columns={NBI_COL: "nbi"})
    return tmp.groupby(["identifiant_tiers", "periode"], as_index=False)["nbi"].sum()

def combine(a, b, val):
    parts = [x for x in (a, b) if x is not None]
    if not parts: return None
    return pd.concat(parts).groupby(["identifiant_tiers", "periode"], as_index=False)[val].sum()

enc = {"epargne": agg_encours(savings, FAMILY_ENCOURS["epargne"][1]),
       "invest": combine(agg_encours(invest, FAMILY_ENCOURS["invest"][1]),
                         agg_encours(banca, FAMILY_ENCOURS["invest_banca"][1]), "encours"),
       "credit_immo": agg_encours(pret_immo, FAMILY_ENCOURS["credit_immo"][1]),
       "autres_cr": agg_encours(credit, FAMILY_ENCOURS["autres_cr"][1])}
nbi = {"epargne": agg_nbi(savings),
       "invest": combine(agg_nbi(invest), agg_nbi(banca), "nbi"),
       "credit_immo": agg_nbi(pret_immo), "autres_cr": agg_nbi(credit), "cartes": agg_nbi(cartes)}

parts = []
for fam in FAM_ORDER:
    n, e = nbi.get(fam), enc.get(fam)
    if n is None and e is None: continue
    base = n.copy() if n is not None else e[["identifiant_tiers", "periode"]].copy()
    if n is None: base["nbi"] = np.nan
    base = base.merge(e, on=["identifiant_tiers", "periode"], how="outer") if e is not None else base.assign(encours=np.nan)
    base["family"] = fam
    parts.append(base)
panel = pd.concat(parts, ignore_index=True).merge(acq, on="identifiant_tiers", how="left").dropna(subset=["acq_period"])
panel["tenure"] = (panel["periode"] - panel["acq_period"]).astype(int)
panel["cohort"] = panel["acq_period"].map(idx_to_year)
panel = panel[panel["tenure"] >= 0]

def eligible_clients(h): return set(max_tenure[max_tenure >= h - 1].index)
def value_at_tenure(values, fam, t):
    p = panel[(panel["family"] == fam) & (panel["tenure"] == t)]
    return p.groupby("identifiant_tiers")[values].sum()
def cum_nbi_client(h, fam=None):
    p = panel if fam is None else panel[panel["family"] == fam]
    return p[p["tenure"] < h].groupby("identifiant_tiers")["nbi"].sum()
print(f"Panel: {len(panel):,} lignes")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MOTEUR DE VENTILATION — 6 caracteristiques client + break_by / count_by
# ══════════════════════════════════════════════════════════════════════════
latest = stock.sort_values("periode").groupby("identifiant_tiers").last()

REV_ORDER = ["Moins de 35.999 EUR", "Entre 36.000 et 53.999 EUR",
             "Entre 54.000 et 119.999 EUR", "Plus de 120.000 EUR"]
AGE_LABELS = ["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]

cli = pd.DataFrame(index=latest.index)
cli["cohorte"] = cohort_of.reindex(cli.index)
cli["canal"]   = latest["canal"] if "canal" in latest.columns else "Inconnu"
rev = latest["revenu_total"].astype(str).str.strip() if "revenu_total" in latest.columns else pd.Series("Inconnu", index=cli.index)
rev = rev.where(rev.isin(REV_ORDER), "Inconnu")
cli["tranche_revenu"] = pd.Categorical(rev, categories=REV_ORDER + ["Inconnu"], ordered=True)
cli["residence"] = (latest["frontalier_o_n"].astype(str).str.strip().str.upper()
                    .map({"O": "Frontalier", "N": "Resident"}).fillna("Inconnu")
                    if "frontalier_o_n" in latest.columns else "Inconnu")
age = pd.to_numeric(latest["age"], errors="coerce") if "age" in latest.columns else pd.Series(np.nan, index=cli.index)
cli["age_num"] = age
cli["tranche_age"] = pd.cut(age, [-1, 17, 24, 34, 44, 54, 64, 200], labels=AGE_LABELS)
cli["csp"] = (latest["segment_ameliore"].astype(str).str.strip()
              if "segment_ameliore" in latest.columns else "Inconnu")

# ── Filtres exercice : hors mineurs / hors etudiants ────────────────────────
EXCLUDE_MINORS   = True    # age < 18 exclus
EXCLUDE_STUDENTS = False   # passe a True si une modalite etudiante existe dans la CSP
student_mask = cli["csp"].str.contains("etud|student", case=False, na=False)
keep = pd.Series(True, index=cli.index)
if EXCLUDE_MINORS:   keep &= (cli["age_num"] >= 18) | cli["age_num"].isna()
if EXCLUDE_STUDENTS: keep &= ~student_mask
cli_all = cli.copy()
cli = cli[keep]

CHARACS = ["cohorte", "canal", "tranche_revenu", "residence", "tranche_age", "csp"]
CHARAC_LABELS = {"cohorte": "Cohorte d'acquisition", "canal": "Canal (EER)",
                 "tranche_revenu": "Tranche de revenu", "residence": "Residence",
                 "tranche_age": "Tranche d'age", "csp": "CSP (segment ameliore)"}
CSP_TOP = 8   # nb de modalites CSP affichees (les plus frequentes)

def _order(ch, s):
    if ch in ("tranche_revenu", "tranche_age"): return s.sort_index()
    if ch == "cohorte": return s.sort_index()
    if ch == "csp": return s.reindex(cli["csp"].value_counts().head(CSP_TOP).index).dropna()
    return s.sort_values(ascending=False)

def count_by(base=None, title="Nombre d'acquisitions par caracteristique", fig_name=None):
    b = cli if base is None else base
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    for ax, ch in zip(axes.ravel(), CHARACS):
        vc = _order(ch, b[ch].value_counts())
        vc.plot(kind="bar", ax=ax, color=PURPLE, edgecolor="white")
        ax.set_title(CHARAC_LABELS[ch]); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
        for i, v in enumerate(vc.values):
            ax.text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=8)
    fig.suptitle(f"{title}  (n={len(b):,} clients)", fontsize=15)
    plt.tight_layout()
    if fig_name: save_fig(fig_name)
    else: plt.show()

def break_by(value, agg="mean", title="", pct=False, fig_name=None, ylabel=""):
    """value: Series indexee par client -> trace agg(value) par caracteristique (grille 2x3)."""
    val = value.reindex(cli.index)
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    for ax, ch in zip(axes.ravel(), CHARACS):
        d = pd.DataFrame({"v": val, "g": cli[ch]}).dropna(subset=["g"])
        g = _order(ch, d.groupby("g", observed=True)["v"].agg(agg)).dropna()
        (g * 100 if pct else g).plot(kind="bar", ax=ax, color=CYAN, edgecolor="white")
        ax.set_title(CHARAC_LABELS[ch]); ax.set_xlabel(""); ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    if fig_name: save_fig(fig_name)
    else: plt.show()

print(f"Clients: {len(cli_all):,} -> {len(cli):,} apres filtres "
      f"(mineurs exclus={EXCLUDE_MINORS}, etudiants exclus={EXCLUDE_STUDENTS})")
print("Modalites CSP (top):", cli['csp'].value_counts().head(CSP_TOP).to_dict())

## 1. Qui sont nos nouveaux clients ?

Distribution des acquisitions selon les 6 caracteristiques : **annee (cohorte), canal
(EER digital vs physique), tranche de revenu, residents/frontaliers, age, CSP**.

In [ ]:
count_by(title="Nombre d'acquisitions par caracteristique", fig_name="s1_acquisitions")

# Table recap chiffree
print("Repartition des acquisitions :")
for ch in CHARACS:
    vc = _order(ch, cli[ch].value_counts())
    print(f"\n{CHARAC_LABELS[ch]} :")
    for k, v in vc.items():
        print(f"  {str(k):<32} {int(v):>8,}  ({v/len(cli)*100:4.1f}%)")